In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Disaster Triage: Single LightGBM Binary Classifier (ESI 1 vs NOT ESI 1) (`models/train_disaster_triage_3class_exp.ipynb`)

Loads the full emergency cohort directly from **`datasets/5v_cleandf.RData`** (~558,000 visits with valid ESI) and trains a single **LightGBM Binary Classifier** to classify whether a patient is **`ESI 1` (Immediate Resuscitation)** or **`NOT ESI 1` (ESI 2–5: Emergent, Urgent, Semi-urgent, Non-urgent)**.

### 🎯 Target Formulation
- **`Class 1: ESI 1 (Immediate Resuscitation)`** ($y=1$): Patients requiring immediate life-saving intervention ($5,271$ visits, $\sim 0.94\%$).
- **`Class 0: NOT ESI 1 (ESI 2–5)`** ($y=0$): Emergent, urgent, and non-urgent visits ($552,758$ visits, $\sim 99.06\%$).

### 🔬 Pipeline Highlights
1. **Direct `5v_cleandf.RData` Loading**: Preserves all 558,029 visits with valid ESI across 16 core clinical vital and demographic features.
2. **Single LightGBM Model**: Plain `LGBMClassifier(objective='binary')` with early stopping (no multi-stage cascades or external calibrators).
3. **Median Imputation**: `SimpleImputer(strategy='median')` fitted strictly on the training partition.
4. **Confusion Matrix Heatmap**: Displays exact visit counts and normalized percentages.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (All Non-NA ESI Rows Kept, No NA Drop)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

# Filter ONLY rows where ESI is not NA
valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df$esi)

# Construct dataframe for 16 core features
df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = raw_esi_char
)

# Export matrices to Python
raw_mat_export <- as.matrix(df_master[, 1:16])
esi_export     <- as.numeric(as.character(df_master$esi))

cat(sprintf("Exported Full Dataset to Python: %d rows, 16 base columns (all NAs preserved for SimpleImputer)\n", nrow(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve from R, Partition & Apply Median Imputation
# ---------------------------------------------------------------------------
import os, json, pickle, warnings
from rpy2.robjects import r
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score, precision_score,
                             f1_score, roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

FEATURES = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp',
    'triage_vital_dbp', 'triage_vital_rr', 'triage_vital_o2'
]

# Binary Target Formulation: 1 = ESI 1 (Immediate Resuscitation), 0 = NOT ESI 1 (ESI 2-5)
y_all = np.where(esi_all == 1, 1, 0)
LABELS = ['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)']

print("=========================================================")
print("         5v_cleandf DATASET BINARY DISTRIBUTION")
print("=========================================================")
print(f"Total Valid ESI Visits: {len(y_all):,}")
print(f"  * Class 0 [NOT ESI 1 (ESI 2-5)]: {np.sum(y_all == 0):,} ({np.mean(y_all == 0)*100:.2f}%)")
print(f"  * Class 1 [ESI 1 (Resuscitation)]: {np.sum(y_all == 1):,} ({np.mean(y_all == 1)*100:.2f}%)")
print(f"Features ({len(FEATURES)}): {FEATURES}")
print("=========================================================\n")

# Stratified 3-way split: 70% Train, 15% Validation, 15% Holdout Test
itr, itmp = train_test_split(np.arange(len(y_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

raw_tr  = raw_mat_all[itr]
raw_val = raw_mat_all[iva]
raw_te  = raw_mat_all[ite]

y_train = y_all[itr]
y_val   = y_all[iva]
y_test  = y_all[ite]

# Fit SimpleImputer strictly on Training set
print("Fitting SimpleImputer(strategy='median') on Training set...")
imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(raw_tr)
X_val   = imputer.transform(raw_val)
X_test  = imputer.transform(raw_te)

print(f"✓ Partition Shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Train Single LightGBM Binary Classifier (ESI 1 vs NOT ESI 1)
# ---------------------------------------------------------------------------
print("Training Single LightGBM Binary Classifier for ESI 1 Classification...")

lgbm_params = {
    'objective': 'binary',
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': 42,
    'verbosity': -1,
    'n_jobs': -1
}

model = LGBMClassifier(**lgbm_params)
model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=True)]
)

dump = model.booster_.dump_model()
n_trees = len(dump['tree_info'])
n_nodes = sum(t['num_leaves'] for t in dump['tree_info'])
print(f"\n✓ Single LightGBM Model Successfully Trained!")
print(f"  * Best Iteration: {model.best_iteration_}")
print(f"  * Total Trees   : {n_trees}")
print(f"  * Total Nodes   : {n_nodes:,} (~{round(n_nodes * 16 / 1024, 2)} KB in .rodata)")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Holdout Test Set Evaluation & Metrics Breakdown
# ---------------------------------------------------------------------------
p_test_esi1 = model.predict_proba(X_test)[:, 1]
pred_test   = model.predict(X_test)

acc      = accuracy_score(y_test, pred_test)
bal_acc  = balanced_accuracy_score(y_test, pred_test)
rec_esi1 = recall_score(y_test == 1, pred_test == 1, zero_division=0)
spec_not = recall_score(y_test == 0, pred_test == 0, zero_division=0)
prec_esi1= precision_score(y_test == 1, pred_test == 1, zero_division=0)
f1_esi1  = f1_score(y_test == 1, pred_test == 1, zero_division=0)
auc_val  = roc_auc_score(y_test, p_test_esi1)
pr_auc   = average_precision_score(y_test, p_test_esi1)

report_data = [{
    'Model': 'Single LightGBM (ESI 1 Classifier)',
    'Accuracy': round(acc, 4),
    'Balanced_Accuracy': round(bal_acc, 4),
    'ESI1_Sensitivity (Recall)': round(rec_esi1, 4),
    'Specificity (NOT ESI 1 Recall)': round(spec_not, 4),
    'ESI1_Precision': round(prec_esi1, 4),
    'ESI1_F1': round(f1_esi1, 4),
    'ROC_AUC': round(auc_val, 4),
    'PR_AUC': round(pr_auc, 4)
}]

report_df = pd.DataFrame(report_data)
print("=========================================================================================================")
print("         HOLDOUT TEST EVALUATION: SINGLE LIGHTGBM (ESI 1 vs NOT ESI 1)")
print("=========================================================================================================")
print(report_df.to_string(index=False))
print("=========================================================================================================\n")

print("Detailed Classification Report:")
print(classification_report(y_test, pred_test, target_names=LABELS, digits=4))

reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'esi1_binary_lightgbm_report.csv')
report_df.to_csv(report_file, index=False)
print(f"Metrics report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Draw the Confusion Matrix Heatmap
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(7.5, 6))
annot = np.empty_like(cm, dtype=object)
for i in range(2):
    for j in range(2):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.2f}%)"

sns.heatmap(
    cm_norm,
    annot=annot,
    fmt='',
    cmap='Blues',
    cbar=True,
    ax=ax,
    vmin=0,
    vmax=1,
    xticklabels=['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)'],
    yticklabels=['NOT ESI 1 (ESI 2-5)', 'ESI 1 (Resuscitation)']
)

ax.set_title(
    f'Single LightGBM: Confusion Matrix (ESI 1 vs NOT ESI 1)\n'
    f'Accuracy: {acc*100:.2f}% | ESI 1 Sensitivity: {rec_esi1*100:.2f}% | Specificity: {spec_not*100:.2f}%',
    fontsize=12,
    fontweight='bold',
    pad=12
)
ax.set_xlabel('Predicted Label', fontsize=11, fontweight='bold')
ax.set_ylabel('True Label', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_plot_path = os.path.join(plots_dir, 'disaster_triage_confusion_matrix.png')
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'disaster_triage_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'esi1_binary_lightgbm_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion Matrix saved to: {cm_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: Feature Importances & ROC / Precision-Recall Curves
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Top Feature Importances
imp_vals = model.feature_importances_ / model.feature_importances_.sum()
imp_df = pd.DataFrame({'Feature': FEATURES, 'Importance': imp_vals}).sort_values(by='Importance', ascending=False).head(10)

sns.barplot(data=imp_df, x='Feature', y='Importance', color='#1f77b4', edgecolor='black', ax=axes[0])
axes[0].set_title('Single LightGBM: Top Feature Importances (ESI 1 vs NOT ESI 1)', fontsize=12.5, fontweight='bold', pad=12)
axes[0].set_xlabel('Feature', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Relative Importance', fontsize=11, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right', fontweight='bold')
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height()*100:.1f}%",
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=9.5, fontweight='bold', xytext=(0, 2),
                     textcoords='offset points')
axes[0].grid(True, linestyle='--', alpha=0.4)

# Panel 2: ROC Curve
fpr, tpr, _ = roc_curve(y_test, p_test_esi1)
axes[1].plot(fpr, tpr, color='#1f77b4', linewidth=2.2, label=f"Single LightGBM (AUC = {auc_val:.4f})")
axes[1].plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Chance (0.5000)')
axes[1].set_title('ROC Curve: ESI 1 vs NOT ESI 1', fontsize=12.5, fontweight='bold', pad=12)
axes[1].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('True Positive Rate (Sensitivity)', fontsize=11, fontweight='bold')
axes[1].legend(loc="lower right", fontsize=10.5, frameon=True, framealpha=0.95)
axes[1].grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
eval_curves_path = os.path.join(plots_dir, 'esi1_binary_lightgbm_eval_curves.png')
plt.savefig(eval_curves_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Evaluation curves saved to: {eval_curves_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: Export Production Bundle & Deployment Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle_data = {
    'model': model,
    'imputer': imputer,
    'features': FEATURES,
    'labels': LABELS
}

bundle_file = os.path.join(deploy_dir, 'disaster_triage_esi1_binary.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle_data, f)

manifest = dict(
    target='Binary_ESI1_vs_NOT_ESI1',
    model_type='Single_LGBMClassifier',
    dataset='5v_cleandf_RData',
    n_trees=n_trees,
    n_nodes=n_nodes,
    best_iteration=int(model.best_iteration_),
    total_visits=len(y_all),
    n_esi1=int(np.sum(y_all == 1)),
    n_not_esi1=int(np.sum(y_all == 0)),
    labels=LABELS,
    feature_order=FEATURES,
    n_features=len(FEATURES),
    holdout_metrics=report_data[0]
)

manifest_file = os.path.join(deploy_dir, 'disaster_triage_esi1_binary_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Exported Bundle:   {bundle_file}")
print(f"✓ Exported Manifest: {manifest_file}")